# 💎 Previsão de Preço de Diamantes com Redes Neurais

Este notebook treina dois modelos Keras (Redes Neurais) para prever o preço de diamantes com base em suas características (4Cs e dimensões). Também implementa uma lógica de Ensemble (Voting) para combinar as previsões.

**Conteúdo:**
1. Carregamento e Exploração dos Dados
2. Visualização e Análise Exploratória
3. Pré-processamento
4. Treinamento dos Modelos
5. Avaliação e Resultados
6. Salvar Modelos

In [ ]:
!pip install tensorflow pandas seaborn scikit-learn joblib numpy matplotlib -q

In [ ]:
import pandas as pd
import seaborn as sns
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
import joblib
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
print(f"TensorFlow Version: {tf.__version__}")

## 1. 📊 Carregamento e Exploração dos Dados

In [ ]:
try:
    df = pd.read_csv('diamonds.csv')
    print("✅ Carregado de diamonds.csv")
except:
    print("📥 Arquivo local não encontrado, carregando do Seaborn...")
    df = sns.load_dataset('diamonds')

print(f"\n📦 Total de diamantes: {len(df):,}")
print(f"📋 Total de características: {len(df.columns)}")
print(f"\n🔍 Primeiros 10 registros:")
df.head(10)

In [ ]:
print("📈 Estatísticas Descritivas:")
df.describe()

In [ ]:
print("🔍 Informações do Dataset:")
df.info()

In [ ]:
print("❓ Valores Nulos por Coluna:")
print(df.isnull().sum())
print(f"\n✅ Total de valores nulos: {df.isnull().sum().sum()}")

## 2. 📉 Visualização e Análise Exploratória

### 2.1 Distribuição do Preço (Variável Alvo)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['price'], bins=50, color='#3498db', edgecolor='white')
axes[0].set_xlabel('Preço ($)')
axes[0].set_ylabel('Frequência')
axes[0].set_title('Distribuição dos Preços dos Diamantes')
axes[0].axvline(df['price'].mean(), color='red', linestyle='--', label=f"Média: ${df['price'].mean():,.0f}")
axes[0].axvline(df['price'].median(), color='green', linestyle='--', label=f"Mediana: ${df['price'].median():,.0f}")
axes[0].legend()

axes[1].boxplot(df['price'], vert=True)
axes[1].set_ylabel('Preço ($)')
axes[1].set_title('Boxplot dos Preços (Outliers)')

plt.tight_layout()
plt.show()

print(f"💰 Preço Mínimo: ${df['price'].min():,}")
print(f"💰 Preço Máximo: ${df['price'].max():,}")
print(f"💰 Preço Médio: ${df['price'].mean():,.2f}")
print(f"💰 Preço Mediano: ${df['price'].median():,}")

### 2.2 Distribuição das Variáveis Categóricas (Corte, Cor, Clareza)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

cut_order = ['Fair', 'Good', 'Very Good', 'Premium', 'Ideal']
color_order = ['J', 'I', 'H', 'G', 'F', 'E', 'D']
clarity_order = ['I1', 'SI2', 'SI1', 'VS2', 'VS1', 'VVS2', 'VVS1', 'IF']

sns.countplot(data=df, x='cut', order=cut_order, palette='Blues_d', ax=axes[0])
axes[0].set_title('Distribuição por Corte')
axes[0].set_xlabel('Corte (Fair=Pior → Ideal=Melhor)')
axes[0].set_ylabel('Quantidade')
axes[0].tick_params(axis='x', rotation=45)

sns.countplot(data=df, x='color', order=color_order, palette='Greens_d', ax=axes[1])
axes[1].set_title('Distribuição por Cor')
axes[1].set_xlabel('Cor (J=Pior → D=Melhor)')
axes[1].set_ylabel('Quantidade')

sns.countplot(data=df, x='clarity', order=clarity_order, palette='Oranges_d', ax=axes[2])
axes[2].set_title('Distribuição por Clareza')
axes[2].set_xlabel('Clareza (I1=Pior → IF=Melhor)')
axes[2].set_ylabel('Quantidade')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

### 2.3 Preço Médio por Categoria

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

df.groupby('cut')['price'].mean().reindex(cut_order).plot(kind='bar', ax=axes[0], color='#3498db', edgecolor='white')
axes[0].set_title('Preço Médio por Corte')
axes[0].set_xlabel('Corte')
axes[0].set_ylabel('Preço Médio ($)')
axes[0].tick_params(axis='x', rotation=45)

df.groupby('color')['price'].mean().reindex(color_order).plot(kind='bar', ax=axes[1], color='#27ae60', edgecolor='white')
axes[1].set_title('Preço Médio por Cor')
axes[1].set_xlabel('Cor')
axes[1].set_ylabel('Preço Médio ($)')

df.groupby('clarity')['price'].mean().reindex(clarity_order).plot(kind='bar', ax=axes[2], color='#e67e22', edgecolor='white')
axes[2].set_title('Preço Médio por Clareza')
axes[2].set_xlabel('Clareza')
axes[2].set_ylabel('Preço Médio ($)')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print("⚠️ Observação: Cortes 'piores' podem ter preço médio maior porque tendem a ter diamantes maiores (mais quilates)!")

### 2.4 🔥 Matriz de Correlação entre Variáveis Numéricas

In [ ]:
numerical_cols_corr = ['carat', 'depth', 'table', 'x', 'y', 'z', 'price']
correlation_matrix = df[numerical_cols_corr].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, 
            fmt='.2f', square=True, linewidths=0.5, 
            cbar_kws={'label': 'Correlação'})
plt.title('Matriz de Correlação entre Variáveis Numéricas', fontsize=14)
plt.tight_layout()
plt.show()

print("\n🔗 Correlação com o Preço (ordenado):")
print(correlation_matrix['price'].sort_values(ascending=False))
print("\n💡 Insight: CARAT (peso) tem a maior correlação com preço (0.92)!")

### 2.5 Relação Carat vs Preço (Scatter Plot)

In [ ]:
plt.figure(figsize=(12, 6))
sample = df.sample(5000, random_state=42)
scatter = plt.scatter(sample['carat'], sample['price'], alpha=0.5, c=sample['price'], cmap='viridis', s=15)
plt.colorbar(scatter, label='Preço ($)')
plt.xlabel('Quilates (Carat)')
plt.ylabel('Preço ($)')
plt.title('Relação entre Quilates e Preço (Amostra de 5.000 diamantes)')
plt.tight_layout()
plt.show()

print("💡 Quanto maior o quilate (peso), maior o preço!")

### 2.6 Distribuição das Variáveis Numéricas

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

num_cols = ['carat', 'depth', 'table', 'x', 'y', 'z']
colors = ['#3498db', '#e74c3c', '#27ae60', '#9b59b6', '#f39c12', '#1abc9c']

for i, col in enumerate(num_cols):
    axes[i].hist(df[col], bins=40, color=colors[i], edgecolor='white')
    axes[i].set_title(f'Distribuição de {col.upper()}')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Frequência')
    axes[i].axvline(df[col].mean(), color='red', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

In [ ]:
## 3. 🔧 Pré-processamento dos Dados

In [ ]:
y = df['price']
X = df.drop('price', axis=1)

categorical_cols = ['cut', 'color', 'clarity']
numerical_cols = ['carat', 'depth', 'table', 'x', 'y', 'z']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols)
    ]
)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"📊 Dados de Treino: {len(X_train):,} diamantes ({len(X_train)/len(df)*100:.0f}%)")
print(f"📊 Dados de Teste: {len(X_test):,} diamantes ({len(X_test)/len(df)*100:.0f}%)")
print(f"🎲 Random State: 42 (dados embaralhados de forma reproduzível)")

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

input_shape = X_train_processed.shape[1]
print(f"\n🔢 Input Shape: {input_shape} features (6 numéricas + 20 categóricas one-hot)")

## 4. 🧠 Definição e Treinamento dos Modelos

In [ ]:
def create_model_1(input_shape):
    model = keras.Sequential([
        layers.Dense(64, activation='relu', input_shape=[input_shape]),
        layers.Dense(32, activation='relu'),
        layers.Dense(1)
    ])
    model.compile(optimizer='adam', loss='mae', metrics=['mae'])
    return model

def create_model_2(input_shape):
    model = keras.Sequential([
        layers.Dense(128, activation='relu', input_shape=[input_shape]),
        layers.Dropout(0.2),
        layers.Dense(64, activation='relu'),
        layers.Dense(32, activation='relu'),
        layers.Dense(1)
    ])
    model.compile(optimizer='adam', loss='mae', metrics=['mae'])
    return model

print("📐 Arquitetura Modelo 1 (Simples):")
model1_temp = create_model_1(input_shape)
model1_temp.summary()

print("\n📐 Arquitetura Modelo 2 (Com Dropout):")
model2_temp = create_model_2(input_shape)
model2_temp.summary()

In [ ]:
print("🚀 Treinando Modelo 1 (Simples)...")
model1 = create_model_1(input_shape)
history1 = model1.fit(X_train_processed, y_train, validation_split=0.2, batch_size=32, epochs=50, verbose=1)

print("\n🚀 Treinando Modelo 2 (Com Dropout)...")
model2 = create_model_2(input_shape)
history2 = model2.fit(X_train_processed, y_train, validation_split=0.2, batch_size=32, epochs=50, verbose=1)

print("\n✅ Treinamento concluído!")

### 4.1 Histórico de Treinamento (Loss por Época)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history1.history['mae'], label='Treino', color='#3498db', linewidth=2)
axes[0].plot(history1.history['val_mae'], label='Validação', color='#e74c3c', linewidth=2)
axes[0].set_xlabel('Época')
axes[0].set_ylabel('MAE ($)')
axes[0].set_title('Modelo 1 - Histórico de Treinamento')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history2.history['mae'], label='Treino', color='#3498db', linewidth=2)
axes[1].plot(history2.history['val_mae'], label='Validação', color='#e74c3c', linewidth=2)
axes[1].set_xlabel('Época')
axes[1].set_ylabel('MAE ($)')
axes[1].set_title('Modelo 2 - Histórico de Treinamento')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("💡 Se as linhas de treino e validação se afastam muito = Overfitting!")
print("💡 Se ambas estão altas e não diminuem = Underfitting!")

## 5. 📊 Avaliação e Voting Ensemble

In [ ]:
loss1, mae1 = model1.evaluate(X_test_processed, y_test, verbose=0)
loss2, mae2 = model2.evaluate(X_test_processed, y_test, verbose=0)

pred1 = model1.predict(X_test_processed, verbose=0).flatten()
pred2 = model2.predict(X_test_processed, verbose=0).flatten()
pred_voting = (pred1 + pred2) / 2

mae_voting = np.mean(np.abs(y_test - pred_voting))

print("="*50)
print("📊 RESULTADOS FINAIS")
print("="*50)
print(f"🔵 Modelo 1 MAE: ${mae1:,.2f}")
print(f"🟢 Modelo 2 MAE: ${mae2:,.2f}")
print(f"🟣 Voting Ensemble MAE: ${mae_voting:,.2f}")
print("="*50)

best_mae = min(mae1, mae2, mae_voting)
if best_mae == mae_voting:
    print("🏆 O Voting Ensemble teve o melhor desempenho!")
elif best_mae == mae1:
    print("🏆 O Modelo 1 teve o melhor desempenho!")
else:
    print("🏆 O Modelo 2 teve o melhor desempenho!")

### 5.1 Comparação: Previsão vs Valor Real

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sample_size = 1000
indices = np.random.choice(len(y_test), sample_size, replace=False)
y_sample = y_test.iloc[indices].values

max_val = max(y_sample.max(), pred1[indices].max())

axes[0].scatter(y_sample, pred1[indices], alpha=0.5, s=10, c='#3498db')
axes[0].plot([0, max_val], [0, max_val], 'r--', lw=2, label='Previsão Perfeita')
axes[0].set_xlabel('Preço Real ($)')
axes[0].set_ylabel('Preço Previsto ($)')
axes[0].set_title(f'Modelo 1 (MAE: ${mae1:,.0f})')
axes[0].legend()

axes[1].scatter(y_sample, pred2[indices], alpha=0.5, s=10, c='#27ae60')
axes[1].plot([0, max_val], [0, max_val], 'r--', lw=2, label='Previsão Perfeita')
axes[1].set_xlabel('Preço Real ($)')
axes[1].set_ylabel('Preço Previsto ($)')
axes[1].set_title(f'Modelo 2 (MAE: ${mae2:,.0f})')
axes[1].legend()

axes[2].scatter(y_sample, pred_voting[indices], alpha=0.5, s=10, c='#9b59b6')
axes[2].plot([0, max_val], [0, max_val], 'r--', lw=2, label='Previsão Perfeita')
axes[2].set_xlabel('Preço Real ($)')
axes[2].set_ylabel('Preço Previsto ($)')
axes[2].set_title(f'Voting Ensemble (MAE: ${mae_voting:,.0f})')
axes[2].legend()

plt.tight_layout()
plt.show()

print("💡 Quanto mais próximo da linha vermelha, melhor a previsão!")

### 5.2 Distribuição dos Erros

In [ ]:
errors1 = y_test - pred1
errors2 = y_test - pred2
errors_voting = y_test - pred_voting

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(errors1, bins=50, color='#3498db', edgecolor='white', alpha=0.7)
axes[0].axvline(0, color='red', linestyle='--', lw=2, label='Erro Zero')
axes[0].set_xlabel('Erro ($)')
axes[0].set_ylabel('Frequência')
axes[0].set_title('Distribuição dos Erros - Modelo 1')
axes[0].legend()

axes[1].hist(errors2, bins=50, color='#27ae60', edgecolor='white', alpha=0.7)
axes[1].axvline(0, color='red', linestyle='--', lw=2, label='Erro Zero')
axes[1].set_xlabel('Erro ($)')
axes[1].set_ylabel('Frequência')
axes[1].set_title('Distribuição dos Erros - Modelo 2')
axes[1].legend()

axes[2].hist(errors_voting, bins=50, color='#9b59b6', edgecolor='white', alpha=0.7)
axes[2].axvline(0, color='red', linestyle='--', lw=2, label='Erro Zero')
axes[2].set_xlabel('Erro ($)')
axes[2].set_ylabel('Frequência')
axes[2].set_title('Distribuição dos Erros - Voting')
axes[2].legend()

plt.tight_layout()
plt.show()

print("💡 A maioria dos erros deve estar próxima de zero (centro)!")

### 5.3 Comparação de MAE entre Modelos

In [ ]:
models_names = ['Modelo 1\n(Simples)', 'Modelo 2\n(Dropout)', 'Voting\n(Ensemble)']
maes = [mae1, mae2, mae_voting]
colors = ['#3498db', '#27ae60', '#9b59b6']

plt.figure(figsize=(10, 6))
bars = plt.bar(models_names, maes, color=colors, edgecolor='white', linewidth=2)
plt.ylabel('MAE - Erro Médio Absoluto ($)', fontsize=12)
plt.title('Comparação do Erro Médio Absoluto (MAE) entre Modelos', fontsize=14)

for bar, mae in zip(bars, maes):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20, 
             f'${mae:,.0f}', ha='center', va='bottom', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print("💡 Quanto menor o MAE, melhor o modelo!")

## 6. 💾 Salvar Modelos

Salva os modelos no formato `.keras` e o preprocessor com `joblib`.

In [ ]:
model1.save("model1.keras")
model2.save("model2.keras")
joblib.dump(preprocessor, "preprocessor.joblib")

print("✅ Arquivos salvos com sucesso!")
print("   📁 model1.keras")
print("   📁 model2.keras")
print("   📁 preprocessor.joblib")
print("\n⬇️ Baixe esses arquivos e coloque na pasta 'models/' do projeto!")

In [ ]:
## 7. 🧪 Teste de Carregamento e Previsão

In [ ]:
loaded_model1 = keras.models.load_model("model1.keras")
loaded_model2 = keras.models.load_model("model2.keras")
loaded_preprocessor = joblib.load("preprocessor.joblib")

print("✅ Modelos carregados com sucesso!")

test_diamond = pd.DataFrame([{
    'carat': 1.0,
    'cut': 'Ideal',
    'color': 'G',
    'clarity': 'VS1',
    'depth': 61.5,
    'table': 55.0,
    'x': 6.5,
    'y': 6.5,
    'z': 4.0
}])

print("\n💎 Diamante de Teste:")
print(test_diamond.to_string(index=False))

test_processed = loaded_preprocessor.transform(test_diamond)
pred1_test = loaded_model1.predict(test_processed, verbose=0)[0][0]
pred2_test = loaded_model2.predict(test_processed, verbose=0)[0][0]
pred_voting_test = (pred1_test + pred2_test) / 2

print(f"\n💰 Previsões:")
print(f"   Modelo 1: ${pred1_test:,.2f}")
print(f"   Modelo 2: ${pred2_test:,.2f}")
print(f"   Voting:   ${pred_voting_test:,.2f}")